In [1]:
import os
import re
import json
import time
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

In [2]:
load_dotenv()

INPUT_FILE  = r"C:\Users\USER\Downloads\alterqwen3_6_results.csv"
OUTPUT_FILE = r"C:\Users\USER\Downloads\alterqwen3_6_results_with_pd.csv"

# ── Regex patterns (tried in order; last match wins per pattern) ──────────────

# A: **[Final|Most Supported] Principal Diagnosis:** text_on_same_line
_PA = re.compile(
    r'\*\*(?:Final\s+|Most Supported\s+)?Principal Diagnosis[:\]]\*\*[:\s]*'
    r'(?!\s*\n\s*\*\*)'            # not followed by newline+bold (those hit PB)
    r'([^\n\*]{3,300})',
    re.IGNORECASE,
)

# B: **[Final|Most Supported] Principal Diagnosis:**\n**text** (text on next line in bold)
_PB = re.compile(
    r'\*\*(?:Final\s+|Most Supported\s+)?Principal Diagnosis[:\]]\*\*[:\s]*\n\s*'
    r'\*\*([^*\n]{3,300})\*\*',
    re.IGNORECASE,
)

# C: ### **Principal Diagnosis** \n **text**   (heading style)
_PC = re.compile(
    r'###\s+\*?\*?Principal Diagnosis\*?\*?\s*\n+\*?\*?([^\n\*]{3,300})',
    re.IGNORECASE,
)

# D: **Principal Diagnosis: text**  (colon + text inside bold span)
_PD = re.compile(
    r'\*\*(?:Final\s+|Most Supported\s+)?Principal Diagnosis:\s*([^*\n]{3,300})\*\*',
    re.IGNORECASE,
)

# E: **bold text** is the most [X] supported principal diagnosis
_PE = re.compile(
    r'\*\*([^*\n]{3,200})\*\*\s+is the most\s+\w*\s*supported principal diagnosis',
    re.IGNORECASE,
)

# F: "the most [X] supported principal diagnosis is **text**" or plain text
_PF = re.compile(
    r'most\s+\w*\s*supported principal diagnosis\s+is\s+\*?\*?([^.*\n]{3,200})',
    re.IGNORECASE,
)

PATTERNS = [_PB, _PC, _PA, _PD, _PE, _PF]   # PB before PA so multiline wins


def extract_with_regex(text: str):
    """Return the LAST match across all patterns, or None if no pattern fires."""
    best = None
    for pat in PATTERNS:
        matches = pat.findall(text)
        if matches:
            candidate = matches[-1].strip().rstrip('*').strip()
            # prefer later occurrence in text (proxy: longer search)
            best = candidate
            break   # stop at the first pattern that fires; patterns ordered by specificity
    return best or None


# quick self-test on a few known strings
_tests = [
    ("**Principal Diagnosis:** Urosepsis (UTI)",        "Urosepsis (UTI)"),
    ("**Most Supported Principal Diagnosis:**\n**Unstable Angina**", "Unstable Angina"),
    ("### **Principal Diagnosis**\n**Metastatic Breast Cancer**",    "Metastatic Breast Cancer"),
    ("**Principal Diagnosis: Tracheomalacia** (congenital)",         "Tracheomalacia"),
    ("**Splenic Hematoma** is the most strongly supported principal diagnosis", "Splenic Hematoma"),
    ("most supported principal diagnosis is **Small Bowel Obstruction**",       "Small Bowel Obstruction"),
]
for text, expected in _tests:
    got = extract_with_regex(text)
    status = "OK" if (got and expected.lower() in got.lower()) else f"FAIL (got: {got!r})"
    print(f"  {status} | expected: {expected!r}")

  OK | expected: 'Urosepsis (UTI)'
  OK | expected: 'Unstable Angina'
  OK | expected: 'Metastatic Breast Cancer'
  OK | expected: 'Tracheomalacia'
  OK | expected: 'Splenic Hematoma'
  OK | expected: 'Small Bowel Obstruction'


In [3]:
API_KEY = os.getenv("DEEPSEEK_API_KEY")
MODEL   = "deepseek-v4-flash"   # fast/cheap flash variant

client = OpenAI(api_key=API_KEY, base_url="https://api.deepseek.com")

EXTRACT_SYSTEM = """You are a clinical NLP extraction assistant.
You will receive the FINAL portion of a model's clinical reasoning output.
Your only job: identify the model's stated principal diagnosis and return it as a short phrase.

Rules:
- Extract exactly what the model concludes, do not infer or add information.
- If the model explicitly names a diagnosis at the end, use that.
- If ambiguous between two candidates, pick the one stated last or most emphatically.
- Strip ICD codes, qualifiers like "(with secondary ...)", footnotes, and coding notes — keep only the core diagnosis name.
- If no clear principal diagnosis is stated, return "UNCLEAR".

Return ONLY valid JSON (no markdown fences):
{"principal_diagnosis": "<short diagnosis phrase>"}"""


def extract_with_llm(tail: str) -> str:
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": EXTRACT_SYSTEM},
                {"role": "user",   "content": tail},
            ],
            max_tokens=128,
            temperature=0.0,
        )
        raw = resp.choices[0].message.content
        clean = re.sub(r"```json|```", "", raw).strip()
        return json.loads(clean)["principal_diagnosis"]
    except Exception as e:
        return f"ERROR: {e}"

In [4]:
df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} rows from {INPUT_FILE}")

predicted_pds = []
methods       = []

for i, row in df.iterrows():
    text = str(row["final_answer"])

    pd_value = extract_with_regex(text)
    if pd_value:
        method = "regex"
    else:
        # Feed the last 1200 chars so the LLM sees the conclusion
        tail     = text[-1200:]
        pd_value = extract_with_llm(tail)
        method   = "llm"
        time.sleep(0.3)   # only sleep when hitting the API

    print(f"[{i+1:>2}/{len(df)}] HADM={row['HADM_ID']} cond={row['condition']:<15} [{method}] {pd_value[:90]}")
    predicted_pds.append(pd_value)
    methods.append(method)

df["predicted_pd"] = predicted_pds
df["pd_method"]    = methods

df.to_csv(OUTPUT_FILE, index=False)

# ── Summary ────────────────────────────────────────────────────────────────────
n_regex = methods.count("regex")
n_llm   = methods.count("llm")
n_err   = sum(1 for v in predicted_pds if str(v).startswith("ERROR"))
print(f"\nDone → {OUTPUT_FILE}")
print(f"  regex extracted : {n_regex}/{len(df)}")
print(f"  llm  extracted  : {n_llm}/{len(df)}")
print(f"  errors          : {n_err}")
print()
print(df[["HADM_ID", "condition", "ground_truth", "predicted_pd", "pd_method"]].to_string(index=False))

Loaded 35 rows from C:\Users\USER\Downloads\alterqwen3_6_results.csv
[ 1/35] HADM=197345 cond=0-shot          [regex] Metastatic Breast Cancer
[ 2/35] HADM=197345 cond=1-shot          [llm] Metastatic Breast Cancer
[ 3/35] HADM=197345 cond=counterfactual  [regex] [Final Answer]
[ 4/35] HADM=197345 cond=negated_hx      [regex] Metastatic Breast Cancer (Refractory Metastatic Breast Carcinoma with Omental Carcinomatos
[ 5/35] HADM=197345 cond=negated_ruled_out [regex] Metastatic Breast Cancer (Advanced/Refractory Metastatic Breast Carcinoma with widespread 
[ 6/35] HADM=197345 cond=negated_hedge   [regex] Metastatic Breast Cancer (with complications of malignant ascites, malignant pleural effus
[ 7/35] HADM=197345 cond=random_masked   [regex] Metastatic Breast Cancer (refractory, with malignant pleural effusion, ascites/omental car
[ 8/35] HADM=176830 cond=0-shot          [regex] Lower Gastrointestinal Bleed (Gastrointestinal Hemorrhage)
[ 9/35] HADM=176830 cond=1-shot          [regex] Lo